# Task W1-01 — 데이터 수집

**Feature ID**: DATA-001
**Sub-plan**: `docs/stage1-subplans/w1-01-data-collection.md`

업비트 KRW-BTC 5년치 일봉 + 4시간봉 다운로드. 타임존 localize (KST → UTC). Parquet freeze + SHA256 해시 기록.

## 검증된 API (Day 0)
- pyupbit 0.2.34: `get_ohlcv_from(ticker, interval, fromDatetime, to, period)`. `to=` 존재. 결과는 naive KST.
- 에러 시 None 반환 → 재시도 wrapper 필수.
- Upbit rate limit: 10 req/sec → `period=0.2` (안전 마진).


In [1]:
import pyupbit
import pandas as pd
import numpy as np
import hashlib
import time
from pathlib import Path
from datetime import datetime, timezone

# 버전 검증
from importlib.metadata import version
print(f"pyupbit: {version('pyupbit')}")
print(f"pandas: {version('pandas')}")


pyupbit: 0.2.34
pandas: 2.3.3


In [2]:
def fetch_with_retry(ticker, interval, start, end, max_retries=5, period=0.2):
    """pyupbit get_ohlcv_from with exponential backoff retry.

    검증된 pyupbit 0.2.34 API.
    반환: pd.DataFrame (naive KST index) or raise RuntimeError.
    """
    for attempt in range(max_retries):
        df = pyupbit.get_ohlcv_from(
            ticker=ticker,
            interval=interval,
            fromDatetime=start,
            to=end,
            period=period,
        )
        if df is not None and not df.empty:
            return df
        wait = 2 ** attempt
        print(f"Attempt {attempt+1} returned None/empty. Sleeping {wait}s...")
        time.sleep(wait)
    raise RuntimeError(f"Failed to fetch {ticker} {interval} after {max_retries} retries")


# 주: check_gaps 범용 헬퍼는 Cell 7에서 inline으로 대체됨 (advertised 범위 기준).
# 후속 노트북에서 재사용 시 start 파라미터 명시 필수.


def sha256_file(path):
    """Compute SHA256 hash of a file in chunks."""
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(65536), b''):
            h.update(chunk)
    return h.hexdigest()


In [3]:
FREEZE_DATE = "20260412"
START = "2021-01-01 00:00:00"
END = "2026-04-12 00:00:00"

print("Downloading KRW-BTC daily candles...")
df_daily = fetch_with_retry(
    ticker="KRW-BTC",
    interval="day",
    start=START,
    end=END,
    period=0.2,
)
print(f"Daily: {len(df_daily)} bars, first={df_daily.index[0]}, last={df_daily.index[-1]}")
df_daily.head()


Daily: 1927 bars, first=2021-01-01 09:00:00, last=2026-04-11 09:00:00


,open,high,low,close,volume,value
2021-01-01 09:00:00,32037000.0,32599000.0,31800000.0,32296000.0,5752.494216,1.856139e+11
2021-01-02 09:00:00,32295000.0,36600000.0,31920000.0,35700000.0,17451.167678,6.023355e+11
2021-01-03 09:00:00,35700000.0,39453000.0,35500000.0,37537000.0,25381.506623,9.588751e+11
2021-01-04 09:00:00,37537000.0,38476000.0,33000000.0,36460000.0,23598.194590,8.472770e+11
2021-01-05 09:00:00,36478000.0,38590000.0,34357000.0,38093000.0,13461.981681,4.915540e+11


In [4]:
print("Downloading KRW-BTC 4h candles (may take ~30s for 5 years)...")
df_4h = fetch_with_retry(
    ticker="KRW-BTC",
    interval="minute240",
    start=START,
    end=END,
    period=0.2,
)
print(f"4h: {len(df_4h)} bars, first={df_4h.index[0]}, last={df_4h.index[-1]}")
df_4h.head()


4h: 11563 bars, first=2021-01-01 01:00:00, last=2026-04-12 05:00:00


,open,high,low,close,volume,value
2021-01-01 01:00:00,32019000.0,32085000.0,31548000.0,32014000.0,468.342616,1.492133e+10
2021-01-01 05:00:00,32005000.0,32200000.0,31899000.0,32042000.0,551.524248,1.770143e+10
2021-01-01 09:00:00,32037000.0,32495000.0,31819000.0,32308000.0,1699.453630,5.486629e+10
2021-01-01 13:00:00,32308000.0,32373000.0,32077000.0,32174000.0,766.626559,2.470116e+10
2021-01-01 17:00:00,32174000.0,32448000.0,31970000.0,32358000.0,1007.097304,3.248881e+10


In [5]:
# pyupbit는 timezone-naive KST 반환 (검증됨).
# 1) KST로 localize → UTC로 convert
# 2) Advertised UTC 범위로 슬라이싱 (사용자 의도: 2021-01-01 00:00 UTC ~ 2026-04-11 23:59 UTC)
#    - pyupbit는 START/END를 KST로 해석하므로 4h 결과가 2020-12-31 16:00 UTC부터 시작함.
#    - 이를 advertised 범위로 정렬해야 data_hashes.txt 헤더와 실제 데이터가 일치.

ADVERTISED_START_UTC = pd.Timestamp('2021-01-01 00:00:00', tz='UTC')
ADVERTISED_END_UTC   = pd.Timestamp('2026-04-12 00:00:00', tz='UTC')  # exclusive upper bound

for name, df in [('daily', df_daily), ('4h', df_4h)]:
    assert df.index.tz is None, f"{name}: Expected naive timestamps from pyupbit"

df_daily.index = df_daily.index.tz_localize('Asia/Seoul').tz_convert('UTC')
df_4h.index = df_4h.index.tz_localize('Asia/Seoul').tz_convert('UTC')

# Slice to advertised range (half-open [START, END))
before_daily = len(df_daily)
before_h4 = len(df_4h)
df_daily = df_daily[(df_daily.index >= ADVERTISED_START_UTC) & (df_daily.index < ADVERTISED_END_UTC)]
df_4h    = df_4h[(df_4h.index >= ADVERTISED_START_UTC) & (df_4h.index < ADVERTISED_END_UTC)]

print(f"Daily: sliced {before_daily} -> {len(df_daily)}, range: {df_daily.index[0]} ~ {df_daily.index[-1]}")
print(f"4h:    sliced {before_h4} -> {len(df_4h)}, range: {df_4h.index[0]} ~ {df_4h.index[-1]}")
print(f"Advertised: {ADVERTISED_START_UTC} ~ {ADVERTISED_END_UTC} (exclusive)")


Daily: sliced 1927 -> 1927, range: 2021-01-01 00:00:00+00:00 ~ 2026-04-11 00:00:00+00:00
4h:    sliced 11563 -> 11561, range: 2021-01-01 00:00:00+00:00 ~ 2026-04-11 20:00:00+00:00
Advertised: 2021-01-01 00:00:00+00:00 ~ 2026-04-12 00:00:00+00:00 (exclusive)


In [6]:
# 갭은 advertised 시작 시점 기준으로 계산 (시작 경계 누락 감지).
# 끝 경계는 df.index[-1] 까지 — freeze 시점에 마지막 bar가 아직 존재하지 않을 수 있기 때문.
# 이 정책은 의도적이며, 만약 끝단 누락도 감지하고 싶으면 end=ADVERTISED_END_UTC-1bar로 수정.

expected_daily = pd.date_range(start=ADVERTISED_START_UTC, end=df_daily.index[-1], freq='1D', tz='UTC')
expected_h4    = pd.date_range(start=ADVERTISED_START_UTC, end=df_4h.index[-1],    freq='4h', tz='UTC')

daily_missing = expected_daily.difference(df_daily.index)
h4_missing    = expected_h4.difference(df_4h.index)

daily_pct = len(daily_missing) / len(expected_daily) * 100
h4_pct    = len(h4_missing)    / len(expected_h4)    * 100

print(f"Daily: expected {len(expected_daily)}, actual {len(df_daily)}, missing {len(daily_missing)} ({daily_pct:.4f}%)")
print(f"4h:    expected {len(expected_h4)},    actual {len(df_4h)},    missing {len(h4_missing)}    ({h4_pct:.4f}%)")

if len(daily_missing) > 0:
    print(f"Daily missing (first 5): {list(daily_missing[:5])}")
if len(h4_missing) > 0:
    print(f"4h missing (first 5): {list(h4_missing[:5])}")

# 갭 > 0.1%면 경고 (룰)
assert daily_pct < 0.1, f"Daily gaps {daily_pct:.4f}% exceed 0.1% threshold"
assert h4_pct    < 0.1, f"4h gaps {h4_pct:.4f}% exceed 0.1% threshold"
print("\nGap check PASSED (both < 0.1%)")


Daily: expected 1927, actual 1927, missing 0 (0.0000%)
4h:    expected 11562,    actual 11561,    missing 1    (0.0086%)
4h missing (first 5): [Timestamp('2023-12-03 16:00:00+0000', tz='UTC')]

Gap check PASSED (both < 0.1%)


In [7]:
data_dir = Path('../data')
data_dir.mkdir(exist_ok=True)

daily_path = data_dir / f'KRW-BTC_1d_frozen_{FREEZE_DATE}.parquet'
h4_path = data_dir / f'KRW-BTC_4h_frozen_{FREEZE_DATE}.parquet'

df_daily.to_parquet(daily_path)
df_4h.to_parquet(h4_path)

print(f"Saved: {daily_path} ({daily_path.stat().st_size / 1024:.1f} KB)")
print(f"Saved: {h4_path} ({h4_path.stat().st_size / 1024:.1f} KB)")


Saved: ../data/KRW-BTC_1d_frozen_20260412.parquet (108.8 KB)
Saved: ../data/KRW-BTC_4h_frozen_20260412.parquet (630.4 KB)


In [8]:
daily_hash = sha256_file(daily_path)
h4_hash = sha256_file(h4_path)

# data_hashes.txt 헤더는 actual 데이터 범위 (advertised 범위 아님)
daily_actual_start = df_daily.index[0].isoformat()
daily_actual_end   = df_daily.index[-1].isoformat()
h4_actual_start    = df_4h.index[0].isoformat()
h4_actual_end      = df_4h.index[-1].isoformat()

hashes_path = data_dir / 'data_hashes.txt'
with open(hashes_path, 'w') as f:
    f.write(f"# Generated by 01_data_collection.ipynb on {datetime.now(timezone.utc).isoformat()}\n")
    f.write(f"# Freeze date: {FREEZE_DATE}\n")
    f.write(f"# Advertised range (UTC, half-open): {ADVERTISED_START_UTC.isoformat()} ~ {ADVERTISED_END_UTC.isoformat()}\n")
    f.write(f"# Daily actual range: {daily_actual_start} ~ {daily_actual_end}\n")
    f.write(f"# 4h actual range:    {h4_actual_start} ~ {h4_actual_end}\n")
    f.write(f"# Daily bars: {len(df_daily)}, gaps: {len(daily_missing)} ({daily_pct:.4f}%)\n")
    f.write(f"# 4h bars: {len(df_4h)}, gaps: {len(h4_missing)} ({h4_pct:.4f}%)\n")
    f.write(f"\n")
    f.write(f"{daily_path.name}: {daily_hash}\n")
    f.write(f"{h4_path.name}: {h4_hash}\n")

print(f"Hashes written to {hashes_path}")
print(f"  {daily_path.name}: {daily_hash}")
print(f"  {h4_path.name}: {h4_hash}")


Hashes written to ../data/data_hashes.txt
  KRW-BTC_1d_frozen_20260412.parquet: da5b5a5bd74c1be06b6c363f71e0f74067008779a554f6a4884733fb066a8504
  KRW-BTC_4h_frozen_20260412.parquet: 0b43e3b86da8534c994beff2e784656fdfc16e7d84db79ef8bc68d5383c29d46


In [9]:
# Final verification summary (printed for evidence file)
print("=" * 60)
print("W1-01 Data Collection Verification")
print("=" * 60)
print(f"Advertised range: {ADVERTISED_START_UTC} ~ {ADVERTISED_END_UTC} (half-open)")
print(f"Daily: {len(df_daily)} bars, gaps {len(daily_missing)} ({daily_pct:.4f}%), hash {daily_hash[:16]}...")
print(f"  range: {df_daily.index[0]} ~ {df_daily.index[-1]}")
print(f"4h:    {len(df_4h)} bars, gaps {len(h4_missing)} ({h4_pct:.4f}%), hash {h4_hash[:16]}...")
print(f"  range: {df_4h.index[0]} ~ {df_4h.index[-1]}")
print(f"TZ: UTC (localized from naive KST)")
print(f"Files: {daily_path.name}, {h4_path.name}")
if len(h4_missing) > 0:
    print(f"\n4h missing timestamps:")
    for ts in h4_missing:
        print(f"  {ts}")


W1-01 Data Collection Verification
Advertised range: 2021-01-01 00:00:00+00:00 ~ 2026-04-12 00:00:00+00:00 (half-open)
Daily: 1927 bars, gaps 0 (0.0000%), hash da5b5a5bd74c1be0...
  range: 2021-01-01 00:00:00+00:00 ~ 2026-04-11 00:00:00+00:00
4h:    11561 bars, gaps 1 (0.0086%), hash 0b43e3b86da8534c...
  range: 2021-01-01 00:00:00+00:00 ~ 2026-04-11 20:00:00+00:00
TZ: UTC (localized from naive KST)
Files: KRW-BTC_1d_frozen_20260412.parquet, KRW-BTC_4h_frozen_20260412.parquet

4h missing timestamps:
  2023-12-03 16:00:00+00:00
